# You do not need the workstation

**Reproducing this project's headline model on a student Colab subscription.**

CSED 505 · week 3, *infrastructure that survives you* · [WashingtonCsed504](https://github.com/TrueRottweiler/WashingtonCsed504)

**Use an A100.** Budget about 90 minutes, comfortably inside a Colab Pro session.

## Two questions, one run

The bottom poster reports 105 trained models on a workstation holding two RTX PRO 6000
cards. That is roughly **$24,000 of graphics cards** before the machine around them, and it
is the kind of number that makes a reader stop reading, because the implicit message is
*this work is not available to you.*

It is. This notebook rebuilds the corpus and retrains the exact headline model here, and the
last cell works out what the **whole 83-GPU-hour project** would cost at Colab prices.

And while it runs, it settles a second question we asserted and never measured.

## The second question

When Patrick asked for a checkpoint rather than retraining it locally, his reasoning was that
*same corpus, same seed, same steps on a different GPU is a different model* — and that under one
tag, nothing would say so. We agreed, wrote it into the record-keeping rules, and **never measured
it.** This measures it.

The workstation produced `yor_64M_62.5k_s0` at **validation loss 2.315** in 40.2 minutes. This
notebook runs the identical recipe — same corpus, same 64M tokens, same 62,500 steps, same seed 0,
same learning rate — on Colab's hardware, and compares.

## Why it cannot fail to produce a finding

There are two ways it can go and both are worth printing:

1. **The vocabulary comes back different.** `prepare_corpus` retrains the BPE from a FineWeb-2
   stream, and if the stream yields different text the fingerprint will not match
   `15abd33de5af`. Then the two runs are not comparable *at all* — which is precisely the trap
   the fingerprint system exists to catch, demonstrated live rather than asserted.
2. **The vocabulary matches and the loss does not.** Then we can finally say how big "a different
   model" actually is, in the same units as the seed spread — and decide whether Patrick's
   caution was worth a 125 MB upload.

**Keep this tab open.** Colab disconnects idle sessions. Everything is written incrementally and
`reuse=True`, so a reconnect resumes rather than restarts.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
%pip install --quiet transformers datasets tokenizers

!git clone --quiet --depth 1 https://github.com/TrueRottweiler/WashingtonCsed504.git
%cd WashingtonCsed504/src/a2-nlp
print('ready')

In [ ]:
# Prepare the Yoruba corpus from FineWeb-2. data/ is not in the repository -- 153 MB of token
# array is not something to keep in git -- so this rebuilds it, which is the part of the
# experiment that tests whether "the same corpus" is even reproducible. Five to ten minutes.
import mlm_api as factory

# The whole CELL on the workstation, not one lucky draw from it. Three seeds, and the spread
# between them is the yardstick any single new run has to be judged against.
REFERENCE = {
    'seeds': [2.3152, 2.4620, 2.5143],       # yor_64M_62.5k_s0 / _s1 / _s2
    'fingerprint': '15abd33de5af',
    'chars': 259_864_169,
    'chars_per_token': 3.7339,
    'minutes': 40.2,
    'card': 'RTX PRO 6000 Blackwell Max-Q',
}

stats = factory.prepare_corpus('yor', lang='yor_Latn')

# A fingerprint hashes the VOCABULARY, not the token array. BPE over 260M characters is stable
# enough that the underlying stream could shift while the 16k vocabulary comes out identical, so
# a matching fingerprint alone does not establish a matching corpus. Patrick's catch; stats.json
# already carries the two numbers that do check it.
fp = stats.get('tokenizer_fingerprint')
chars = stats.get('chars')
cpt = stats.get('chars_per_token')

print(f"{'':22}{'here':>16}{'workstation':>16}")
print(f"{'vocabulary hash':22}{str(fp):>16}{REFERENCE['fingerprint']:>16}")
print(f"{'characters':22}{chars:>16,}{REFERENCE['chars']:>16,}")
print(f"{'chars per token':22}{cpt:>16.4f}{REFERENCE['chars_per_token']:>16.4f}")

SAME_VOCAB = fp == REFERENCE['fingerprint']
SAME_TEXT = (chars is not None
             and abs(chars - REFERENCE['chars']) / REFERENCE['chars'] < 0.001
             and abs(cpt - REFERENCE['chars_per_token']) < 0.001)
COMPARABLE = SAME_VOCAB and SAME_TEXT

print()
if COMPARABLE:
    print('Same vocabulary AND same corpus. The run below is comparable.')
elif SAME_VOCAB:
    print('Vocabulary matches but the CORPUS DOES NOT -- the stream rebuilt to a different\n'
          'amount of text under an identical 16k vocabulary. That is the finding: a fingerprint\n'
          'is necessary and not sufficient, and the loss below is not comparable.')
else:
    print('Different vocabulary. The runs are not comparable at all, which is what the\n'
          'fingerprint is for. Report the mismatch and ignore the loss below.')


In [ ]:
# The run. Identical arguments to the workstation's, which is the whole point -- if anything here
# is edited, the comparison stops meaning anything.
import time

t0 = time.time()
rec = factory.pretrain('yor', tokens=64_000_000, steps=62_500, seed=0,
                       preset='poc', lr=5e-4, tag='colab_yor_64M_62.5k_s0')
print(f"\nfinished in {(time.time()-t0)/60:.1f} min")

In [ ]:
# The comparison, and the block to send back.
import json, platform, statistics as st

import torch

seeds = REFERENCE['seeds']
lo, hi = min(seeds), max(seeds)
cell_mean, cell_sd = st.mean(seeds), st.stdev(seeds)
here = rec['val_loss']

# Judge against the CELL, not against one draw from it.
#
# The first version of this compared against seed 0 -- 2.3152, the lowest of the three -- and
# called any gap hardware. Under the null that hardware is irrelevant, a Colab run is simply a
# fourth draw from the same distribution, so the expected gap to the luckiest seed is already
# 0.129 before hardware does anything, and the rule printed "hardware matters" 56.6% of the time
# whether it did or not. A test that confirms a hypothesis more often than not, regardless of the
# truth, is not a test. Patrick caught it; the arithmetic is in the commit message.
#
# Note the asymmetry that remains and cannot be removed by one run: landing inside the range
# FAILS TO DETECT a difference, which is not the same as establishing agreement.
INSIDE = lo <= here <= hi

out = {
    'label': 'Colab A100',
    'gpu': torch.cuda.get_device_name(0),
    'torch': torch.__version__,
    'platform': platform.platform(),
    'corpus_comparable': COMPARABLE,
    'vocab_matched': SAME_VOCAB,
    'val_loss_here': round(here, 6),
    'workstation_cell': seeds,
    'workstation_mean': round(cell_mean, 4),
    'workstation_sd': round(cell_sd, 4),
    'inside_cell_range': INSIDE,
    'z_vs_cell': round((here - cell_mean) / cell_sd, 2),
    'minutes_here': round(rec['seconds'] / 60, 1),
    'minutes_workstation': REFERENCE['minutes'],
    'tokens_per_s': round(rec['tokens_per_s']),
}

print('-' * 68)
print(json.dumps(out, indent=2))
print('-' * 68)
print(f"\nworkstation cell: {lo:.4f} .. {hi:.4f}   (mean {cell_mean:.4f}, sd {cell_sd:.4f})")
print(f"here:             {here:.4f}   z = {out['z_vs_cell']:+.2f}")
print()
if not COMPARABLE:
    print('The corpus did not rebuild identically, so the loss comparison is void. Report that.')
elif INSIDE:
    print('INSIDE the range the workstation seeds already span. This FAILS TO DETECT a hardware\n'
          'effect -- which is not the same as showing there is none. One run cannot establish\n'
          'agreement, only fail to find disagreement.')
else:
    print('OUTSIDE the range three workstation seeds span. Suggestive that hardware moves the\n'
          'result by more than reseeding does -- on one draw, so suggestive is all it is.')
print()
print('And the limit of this test, which is Patrick\'s point and the sharper one:')
print('  a matching validation loss would still not license swapping checkpoints inside the')
print('  0.632 downstream comparison, because our own correlation study says loss does not')
print('  stand in for downstream score -- 0.79 nats buys 0.044 on SIB and nothing on NER.')
print('  The test that settles it is downstream: fine-tune both checkpoints on SIB-200, one')
print('  learning rate, three seeds. Four minutes, once both checkpoints exist.')


In [ ]:
# What the whole project would cost here instead of on the workstation.
#
# Every rate below is somebody else's pricing page and they all change. They are constants at the
# top of this cell so a reader can correct them, rather than trusting a number baked into a
# poster -- the same discipline as the staleness check in the main notebook, pointed at Google.
PROJECT_GPU_HOURS = 83.3          # what A2 consumed, measured from our own run records
BUDGET_USD = 500                  # the question: does the project fit inside this?
UNITS_PER_HOUR = 11.8             # Colab A100 -- CHECK, this changes
USD_PER_100_UNITS = 9.99          # pay-as-you-go compute units -- CHECK
WORKSTATION_CARDS_USD = 24_000    # two RTX PRO 6000 Blackwell Max-Q

# Our sustained medians, 96 and 55 completed runs. The project is a mix of both model sizes, so
# scaling by only the small one would flatter the answer.
OURS = {'poc': 381_817, 'afriberta': 184_329}
SHARE = {'poc': 0.55, 'afriberta': 0.45}          # roughly how the 83 hours split

here = rec['tokens_per_s']                         # measured above, on the 33.8M model
ratio_small = OURS['poc'] / here
# The larger model is ~2.07x the cost per token on every card we have measured, so assume the
# ratio carries. Stated rather than hidden, because it is the one unmeasured step here.
ratio_big = ratio_small

hours = PROJECT_GPU_HOURS * (SHARE['poc'] * ratio_small + SHARE['afriberta'] * ratio_big)
units = hours * UNITS_PER_HOUR
cost = units / 100 * USD_PER_100_UNITS

print(f'this GPU runs the small model at {here:,.0f} tok/s, '
      f'{ratio_small:.2f}x the workstation')
print(f'the whole project here: {hours:.0f} GPU-hours '
      f'~= {units:,.0f} compute units ~= ${cost:,.0f}')
print()
if cost <= BUDGET_USD:
    print(f'FITS in ${BUDGET_USD}. You could reproduce all 105 models for '
          f'{cost/BUDGET_USD:.0%} of that budget,')
    print(f'against ${WORKSTATION_CARDS_USD:,} of cards -- '
          f'{WORKSTATION_CARDS_USD/max(cost,1):,.0f}x cheaper.')
else:
    print(f'DOES NOT fit in ${BUDGET_USD}; it is ${cost:,.0f}. '
          f'${BUDGET_USD} buys {BUDGET_USD/max(cost,1):.0%} of the project,')
    print('which is still every experiment that matters if you drop the seed replication.')
print()
print('The caveats, because this is the number people will quote:')
print('  - a subscription buys a QUEUE, not a machine. Sessions end, and the 34-hour studies')
print('    in this project would have to be cut into resumable pieces.')
print('  - you get whichever GPU is free, so a study split across an A100 and an L4 has')
print('    hardware as an uncontrolled variable -- which is what the fingerprint check above')
print('    is a demonstration of.')
print('  - owning still wins eventually: the crossover against rental is ~9,300 GPU-hours.')
print('    This project used 83, which is 0.9% of the way there.')


## If the session drops

Re-run every cell. `prepare_corpus` and `pretrain` both check for completed work first, so a
reconnect picks up where it stopped instead of paying for it twice — the same property that makes
the notebooks on the workstation cheap to re-run.

If the runtime is reassigned to a *different* GPU on reconnect, say so when you send the numbers.
Half a run on an A100 and half on an L4 is a third condition, not either of the two.